#03_gold_kpi_churn_admin.sql

Gold - KPI churn administrativo (inadimplência)

##Objetivo
-Entrega para BI inadimplência e risco de churn administrativo por mês e recortes.

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria tabela kpi_churn_admin

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.gold.kpi_churn_admin (
  competencia STRING,
  uf STRING,
  segmento_vinculo STRING,

  vidas_expostas BIGINT,
  vidas_inadimplentes BIGINT,
  pct_inadimplencia DOUBLE,

  valor_faturado_total DECIMAL(18,2),
  valor_pago_total DECIMAL(18,2),
  gap_pagamento DECIMAL(18,2),

  gold_build_ts TIMESTAMP
)
USING DELTA;

##Constroe dataset gold

In [0]:
INSERT OVERWRITE healthcare_dev.gold.kpi_churn_admin
WITH base AS (
  SELECT
    competencia,
    uf,
    segmento_vinculo,
    beneficiario_id,
    flag_inadimpencia,
    valor_faturado,
    valor_pago
  FROM healthcare_dev.gold.mart_member_month
)
SELECT
  competencia,
  uf,
  segmento_vinculo,
  COUNT(DISTINCT beneficiario_id) AS vidas_expostas,
  SUM(CASE WHEN flag_inadimpencia = 1 THEN 1 ELSE 0 END) AS vidas_inadimplentes,
  ROUND(100.0 * SUM(CASE WHEN flag_inadimpencia = 1 THEN 1 ELSE 0 END) / COUNT(DISTINCT beneficiario_id), 4) AS pct_inadimplencia,
  SUM(valor_faturado) AS valor_faturado_total,
  SUM(valor_pago) AS valor_pago_total,
  SUM(valor_faturado) - SUM(valor_pago) AS gap_pagamento,
  current_timestamp() AS gold_build_ts
FROM base
GROUP BY competencia, uf, segmento_vinculo;